In [ ]:
from collections import defaultdict
import cv2
import numpy as np
from ultralytics import YOLO
import csv
import os
from collections import defaultdict, Counter

ENTRY_MARGIN = 0.20
EXIT_MARGIN = 0.10
ABSENT_THRESHOLD = 5  # How many consecutive frames a track must be absent before we declare it "gone"
MIN_TRACK_LENGTH = 5   # Minimum track length to be considered for counting



def get_zone(cx, frame_width, margin_ratio=ENTRY_MARGIN):
    if cx < frame_width * margin_ratio:               # left boundary
        return "left"
    elif cx > frame_width * (1 - margin_ratio):       # right boundary
        return "right"
    else:
        return "middle"
    
def is_valid_traversal(track, frame_width):
    """
    Decide whether a track represents a valid traversal (in either direction).
    Returns True if it should be counted, False otherwise.
    """
    # Filter out very short tracks (likely noise)
    track_length = track["last_frame"] - track["first_frame"]
    if track_length < MIN_TRACK_LENGTH:
        return False

    first_side = track["first_side"]
    last_side = track["last_side"]
    final_x = track["positions"][-1][0]

    exited_right = final_x > frame_width * (1 - EXIT_MARGIN)
    exited_left = final_x < frame_width * EXIT_MARGIN

    # Valid traversal: entered one side, exited the opposite side
    if first_side == "left" and last_side == "right" and exited_right:
        return True
    if first_side == "right" and last_side == "left" and exited_left:
        return True

    return False

def decide_track_class(class_history):
    """Majority vote across the track's per-frame class votes."""
    return Counter(class_history).most_common(1)[0][0]


def fish_counting_driver(model, input_video_path:str, output_video_path:str, output_csv_path:str):

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {input_video_path}")

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # State
    tracks = {}                     # active + recently-disappeared tracks
    counted_ids = set()             # tracks already counted
    fish_counts = defaultdict(int)  # count of fish by class
    id_remap = {}                   # map old track IDs to new canonical IDs


    # Open CSV file for writing
    csv_file = open(output_csv_path, mode='w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow([
        "frame_id",
        "track_id",
        "class_name",
        "confidence",
        "center_x", "center_y",
        "status",
        "track_info"
    ])

    frame_id = -1

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        frame_id += 1

        # --- run detection + tracking ---
        results = model.track(frame, persist=True, tracker="my_botsort.yaml", verbose=False)

        current_frame_ids = set()

        # --- annotated frame for visualization ---
        if results[0].boxes is not None and results[0].boxes.id is not None:
            annotated_frame = frame.copy()
            xywh = results[0].boxes.xywh.cpu().tolist()
            ids = results[0].boxes.id.int().cpu().tolist()
            confidences = results[0].boxes.conf.cpu().tolist()
            class_ids = results[0].boxes.cls.int().cpu().tolist()

            # --- update track states ---
            for (x_c, y_c, w, h), raw_tid, conf, cid in zip(xywh, ids, confidences, class_ids):
                cname = model.names[cid]
                current_side = get_zone(x_c, width)

                tid = id_remap.get(raw_tid, raw_tid)

                if tid not in tracks:
                    tracks[tid] = {
                        "first_frame": frame_id,
                        "last_frame": frame_id,
                        "first_side": current_side,
                        "last_side": current_side,
                        "positions": [(x_c, y_c)],
                        "class_history": [cname],
                        "absent_frames": 0,
                        "status": "tracking",
                    }

                else:
                    t = tracks[tid]
                    t["last_frame"] = frame_id
                    t["last_side"] = current_side
                    t["positions"].append((x_c, y_c))
                    t["class_history"].append(cname)
                    t["absent_frames"] = 0

                    # if it reappears after missing, make it tracking again
                    if t["status"] == "missing":
                        t["status"] = "tracking"

                current_frame_ids.add(tid)
                track = tracks[tid]

                # Change bbox color if fish first appeared in middle
                if track["first_side"] == "middle":
                    box_color = (150, 150, 150)      # gray
                else:
                    box_color = (255, 0, 0)      # blue

                x1 = int(x_c - w / 2)
                y1 = int(y_c - h / 2)
                x2 = int(x_c + w / 2)
                y2 = int(y_c + h / 2)

                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), box_color, 2)

                # show stitch info in label when remapped
                if raw_tid != tid:
                    label = f"ID:{tid}(<-{raw_tid}) {cname} {conf:.2f}"
                else:
                    label = f"ID:{tid} {cname} {conf:.2f}"
                cv2.putText(
                    annotated_frame,
                    label,
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    box_color,
                    2
                )

                track_info = (
                    f"first_frame={track['first_frame']}\n"
                    f"last_frame={track['last_frame']}\n"
                    f"first_side={track['first_side']}\n"
                    f"last_side={track['last_side']}\n"
                    f"class_history={track['class_history']}\n"
                    f"absent_frames={track['absent_frames']}"
                )

                # ─── Real-time counting check ────────
                if tid not in counted_ids and is_valid_traversal(track, width):
                    final_class = decide_track_class(track["class_history"])
                    fish_counts[final_class] += 1
                    counted_ids.add(tid)
                    track["status"] = f"counted_{final_class}"


                # --- log each detection to CSV ---
                csv_writer.writerow([
                    frame_id, 
                    tid, 
                    cname, 
                    f"{conf:.3f}",
                    f"{x_c:.2f}", f"{y_c:.2f}",
                    track["status"],
                    track_info
                ])

        else:
            annotated_frame = frame.copy()
            xywh, ids, confidences, class_ids = [], [], [], []

        # Update absent counters ONCE per frame, outside the detection loop
        stale_noise = []
        for missing_tid, track in tracks.items():
            if missing_tid not in current_frame_ids and missing_tid not in counted_ids:
                track["absent_frames"] += 1
                if track["absent_frames"] >= ABSENT_THRESHOLD:
                    track["status"] = "missing"

        # Delete noise tracks (also clean up any remap entries pointing to them)
        for tid in stale_noise:
            del tracks[tid]
        if stale_noise:
            stale_set = set(stale_noise)
            for raw, canonical in list(id_remap.items()):
                if canonical in stale_set or raw in stale_set:
                    del id_remap[raw]

         # ─── Draw zone boundaries on the annotated frame ─────────────────────
        left_line = int(width * ENTRY_MARGIN)
        right_line = int(width * (1 - ENTRY_MARGIN))
        cv2.line(annotated_frame, (left_line, 0), (left_line, height), (255, 255, 0), 2)
        cv2.line(annotated_frame, (right_line, 0), (right_line, height), (255, 255, 0), 2)
        
        # Draw counted IDs top left        
        max_ids_per_line = 20
        y_offset = 40
        for i in range(0, len(counted_ids), max_ids_per_line):
            line_ids = ", ".join(map(str, sorted(counted_ids)[i:i+max_ids_per_line]))

            if i == 0:
                line = "Counted IDs: " + line_ids
                x = 20
                y = y_offset
            else:
                line = line_ids
                x = 40
                y = y_offset + (i // max_ids_per_line) * 30

            cv2.putText(
                annotated_frame,
                line,
                (x, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 255),
                2
            )
        
        # Draw entered IDs top left
        entered_list = [
            tid for tid in tracks
            if tracks[tid]["status"] == "tracking"
            and len(tracks[tid]["positions"]) >= MIN_TRACK_LENGTH
        ]

        entered_y_offset = y_offset + ((len(counted_ids) - 1) // max_ids_per_line + 1) * 30 + 40

        for i in range(0, len(entered_list), max_ids_per_line):
            line_ids = ", ".join(map(str, entered_list[i:i+max_ids_per_line]))

            if i == 0:
                line = "Entered IDs: " + line_ids
                x = 20
                y = entered_y_offset
            else:
                line = line_ids
                x = 40
                y = entered_y_offset + (i // max_ids_per_line) * 30

            cv2.putText(
                annotated_frame,
                line,
                (x, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 0),
                2
            )

        # Draw missing IDs below entered IDs
        missing_list = [
            tid for tid in tracks
            if tracks[tid]["status"] == "missing"
            and len(tracks[tid]["positions"]) >= MIN_TRACK_LENGTH
        ]

        missing_y_offset = entered_y_offset + ((len(entered_list) - 1) // max_ids_per_line + 1) * 30 + 40

        for i in range(0, len(missing_list), max_ids_per_line):
            line_ids = ", ".join(map(str, missing_list[i:i+max_ids_per_line]))

            if i == 0:
                line = "Missing IDs: " + line_ids
                x = 20
                y = missing_y_offset
            else:
                line = line_ids
                x = 40
                y = missing_y_offset + (i // max_ids_per_line) * 30

            cv2.putText(
                annotated_frame,
                line,
                (x, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

        # Draw counts top right
        herring_count = fish_counts.get("Herring", 0)
        non_herring_count = sum(count for cname, count in fish_counts.items()
                                if cname != "Herring")

        cv2.putText(annotated_frame, f"Herring: {herring_count} | Frame: {frame_id}",
                    (width - 350, 40), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 255, 0), 2)

        cv2.putText(annotated_frame, f"Non-herring: {non_herring_count}",
                    (width - 350, 75), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 255, 0), 2)

        out.write(annotated_frame)



    cap.release()
    out.release()
    csv_file.close()

    print("Done.")




input_video_path="/Users/yangseongwon/Desktop/River_Herring_Counting/Raw_Videos/Maine_DMR_Videos/IMG_2485_1.mov"

output_path_prefix="/Users/yangseongwon/Desktop/River_Herring_Counting/without_tracking_prediction/IMG_2485_1_without_tracking_prediction"
output_video_path=f"{output_path_prefix}.mp4"
output_csv_path=f"{output_path_prefix}.csv"

yolo_model_path="/Users/yangseongwon/Desktop/River_Herring_Counting/Fish-detection-and-counting/model/train113_weights.pt"
yolo_model = YOLO(yolo_model_path)  # load a custom models

# NOTE: use mp4 video fromat, or change the codec accordingly!
fish_counting_driver(
    yolo_model, 
    input_video_path, 
    output_video_path, 
    output_csv_path
    )